In [1]:
import pandas as pd
import numpy as np

# Load the EDA dataset
df = pd.read_csv(
    "../data/traffic_5grids_eda.csv",
    parse_dates=["datetime", "local_datetime"]
)

# Sort data chronologically for each grid
df = df.sort_values(
    ["GridID", "datetime"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head())

Dataset shape: (21600, 7)

Columns: ['GridID', 'TimeInterval', 'internet', 'datetime', 'local_datetime', 'hour', 'date']


,GridID,TimeInterval,internet,datetime,local_datetime,hour,date
0,1,1383260400000,11.028366,2013-10-31 23:00:00+00:00,2013-11-01 00:00:00+01:00,0,2013-11-01
1,1,1383261000000,11.127101,2013-10-31 23:10:00+00:00,2013-11-01 00:10:00+01:00,0,2013-11-01
2,1,1383261600000,10.892771,2013-10-31 23:20:00+00:00,2013-11-01 00:20:00+01:00,0,2013-11-01
3,1,1383262200000,8.622424,2013-10-31 23:30:00+00:00,2013-11-01 00:30:00+01:00,0,2013-11-01
4,1,1383262800000,8.009928,2013-10-31 23:40:00+00:00,2013-11-01 00:40:00+01:00,0,2013-11-01


In [2]:
# Create lag features for each geographical grid
for lag in [1, 2, 3, 6, 144]:
    df[f"lag_{lag}"] = (
        df.groupby("GridID")["internet"].shift(lag)
    )

print("Dataset shape:", df.shape)

display(
    df[df["GridID"] == 1]
    .iloc[144:150][
        ["internet", "lag_1", "lag_2",
         "lag_3", "lag_6", "lag_144"]
    ]
)

Dataset shape: (21600, 12)


,internet,lag_1,lag_2,lag_3,lag_6,lag_144
144,7.837850,13.228100,7.957519,9.272482,8.635192,11.028366
145,8.621555,7.837850,13.228100,7.957519,10.659124,11.127101
146,7.825424,8.621555,7.837850,13.228100,9.063345,10.892771
147,7.964082,7.825424,8.621555,7.837850,9.272482,8.622424
148,6.500324,7.964082,7.825424,8.621555,7.957519,8.009928
149,7.760148,6.500324,7.964082,7.825424,13.228100,8.118420


In [3]:
# Create rolling features separately for each grid
for window in [6, 144]:

    df[f"rolling_mean_{window}"] = (
        df.groupby("GridID")["internet"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    df[f"rolling_std_{window}"] = (
        df.groupby("GridID")["internet"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

print("Dataset shape:", df.shape)

display(
    df[df["GridID"] == 1]
    .iloc[144:150][
        [
            "internet",
            "rolling_mean_6",
            "rolling_std_6",
            "rolling_mean_144",
            "rolling_std_144"
        ]
    ]
)

Dataset shape: (21600, 16)


,internet,rolling_mean_6,rolling_std_6,rolling_mean_144,rolling_std_144
144,7.837850,9.802627,1.900651,10.465614,3.320239
145,8.621555,9.669737,2.022566,10.443457,3.327096
146,7.825424,9.330142,1.994077,10.426058,3.330046
147,7.964082,9.123822,2.088985,10.404757,3.336843
148,6.500324,8.905755,2.138077,10.400185,3.339752
149,7.760148,8.662889,2.340509,10.389701,3.349662


In [4]:
# Extract time features using Milan local time
df["hour"] = df["local_datetime"].dt.hour

df["day_of_week"] = df["local_datetime"].dt.dayofweek

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Extract the 10-minute interval within each hour (0-5)
df["minute_slot"] = df["local_datetime"].dt.minute // 10

print("Dataset shape:", df.shape)

display(
    df[
        ["GridID", "local_datetime", "hour",
         "day_of_week", "is_weekend", "minute_slot"]
    ].head(10)
)

Dataset shape: (21600, 19)


,GridID,local_datetime,hour,day_of_week,is_weekend,minute_slot
0,1,2013-11-01 00:00:00+01:00,0,4,0,0
1,1,2013-11-01 00:10:00+01:00,0,4,0,1
2,1,2013-11-01 00:20:00+01:00,0,4,0,2
3,1,2013-11-01 00:30:00+01:00,0,4,0,3
4,1,2013-11-01 00:40:00+01:00,0,4,0,4
5,1,2013-11-01 00:50:00+01:00,0,4,0,5
6,1,2013-11-01 01:00:00+01:00,1,4,0,0
7,1,2013-11-01 01:10:00+01:00,1,4,0,1
8,1,2013-11-01 01:20:00+01:00,1,4,0,2
9,1,2013-11-01 01:30:00+01:00,1,4,0,3


In [5]:
# Check missing values before cleaning
print("Missing values before cleaning:")
print(df.isnull().sum())

# Remove rows without sufficient historical data
df = df.dropna().reset_index(drop=True)

print("\nDataset shape after cleaning:", df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum().sum())

display(df.head())

Missing values before cleaning:
GridID                0
TimeInterval          0
internet              0
datetime              0
local_datetime        0
hour                  0
date                  0
lag_1                 5
lag_2                10
lag_3                15
lag_6                30
lag_144             720
rolling_mean_6       30
rolling_std_6        30
rolling_mean_144    720
rolling_std_144     720
day_of_week           0
is_weekend            0
minute_slot           0
dtype: int64

Dataset shape after cleaning: (20880, 19)

Remaining missing values:
0


,GridID,TimeInterval,internet,datetime,local_datetime,hour,date,lag_1,lag_2,lag_3,lag_6,lag_144,rolling_mean_6,rolling_std_6,rolling_mean_144,rolling_std_144,day_of_week,is_weekend,minute_slot
0,1,1383346800000,7.837850,2013-11-01 23:00:00+00:00,2013-11-02 00:00:00+01:00,0,2013-11-02,13.228100,7.957519,9.272482,8.635192,11.028366,9.802627,1.900651,10.465614,3.320239,5,1,0
1,1,1383347400000,8.621555,2013-11-01 23:10:00+00:00,2013-11-02 00:10:00+01:00,0,2013-11-02,7.837850,13.228100,7.957519,10.659124,11.127101,9.669737,2.022566,10.443457,3.327096,5,1,1
2,1,1383348000000,7.825424,2013-11-01 23:20:00+00:00,2013-11-02 00:20:00+01:00,0,2013-11-02,8.621555,7.837850,13.228100,9.063345,10.892771,9.330142,1.994077,10.426058,3.330046,5,1,2
3,1,1383348600000,7.964082,2013-11-01 23:30:00+00:00,2013-11-02 00:30:00+01:00,0,2013-11-02,7.825424,8.621555,7.837850,9.272482,8.622424,9.123822,2.088985,10.404757,3.336843,5,1,3
4,1,1383349200000,6.500324,2013-11-01 23:40:00+00:00,2013-11-02 00:40:00+01:00,0,2013-11-02,7.964082,7.825424,8.621555,7.957519,8.009928,8.905755,2.138077,10.400185,3.339752,5,1,4


In [6]:
# Save the feature-engineered dataset
df.to_csv(
    "../data/traffic_features.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")
print("Final shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())

Feature-engineered dataset saved successfully!
Final shape: (20880, 19)
Total missing values: 0
